# Test: GPT-4o — TAMAS Benchmark Baseline (API)

OpenAI API-based. Used for reproducing TAMAS baselines on flat AutoGen.

**Prerequisites:** `OPENAI_API_KEY` set in environment or `.env`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / '.env')

from models.utils import GPT4o

model = GPT4o()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'gpt-4o',
 'provider': 'openai',
 'role': 'tamas_baseline',
 'api_based': True,
 'estimated_cost': '$25-50'}

## 1. Health Check

In [2]:
assert model.health_check(), 'API key invalid or API unreachable!'
print('Health check passed — API key valid')

Health check passed — API key valid


## 2. Chat Completions

In [3]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 2a. Basic chat
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

Basic chat: A lateral movement attack is a technique used by cyber attackers to move through a network, gaining access to additional systems and data after initially compromising a single point.


In [4]:
# 2b. Deterministic
resp1 = model.chat_deterministic(messages, max_tokens=100)
resp2 = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic r1:', resp1.choices[0].message.content[:80])
print('Deterministic r2:', resp2.choices[0].message.content[:80])
print('Match:', resp1.choices[0].message.content == resp2.choices[0].message.content)

Deterministic r1: A lateral movement attack is a technique used by cyber attackers to move through
Deterministic r2: A lateral movement attack is a technique used by cyber attackers to move through
Match: False


In [5]:
# 2c. Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

Creative: A lateral movement attack is a technique used by cyber attackers to move through a network, gaining access to additional systems and data after initially compromising one device.


In [6]:
# 2d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: A lateral movement attack is a technique used by cyber attackers to move through a network, gaining access to additional systems and data after initially compromising a single point.


## 3. Tool / Function Calling

In [7]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs for security events',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Search query'},
                    'time_range': {'type': 'string', 'description': 'Time range'},
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'isolate_host',
            'description': 'Isolate a host from the network',
            'parameters': {
                'type': 'object',
                'properties': {
                    'hostname': {'type': 'string'},
                    'reason': {'type': 'string'},
                },
                'required': ['hostname', 'reason']
            }
        }
    }
]

In [8]:
# 3a. Auto tool choice
tc_messages = [
    {'role': 'system', 'content': 'You are a SOC analyst.'},
    {'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12 in the last hour.'}
]
resp = model.tool_call(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Auto: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Auto: 1 call(s)
  query_siem({"query":"failed logins from 10.0.5.12","time_range":"last hour"})


In [9]:
# 3b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Required: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Required: 1 call(s)
  query_siem({"query":"failed logins from 10.0.5.12","time_range":"last hour"})


In [10]:
# 3c. Specific tool
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
tc = resp.choices[0].message.tool_calls
print(f'Specific: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Specific: 1 call(s)
  isolate_host({"hostname":"10.0.5.12","reason":"Multiple failed login attempts detected, potential security threat."})


## 4. Structured Output

In [11]:
# 4a. JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {
  "severity": "high",
  "category": "unauthorized access attempt",
  "confidence": 0.95
}


In [12]:
# 4b. JSON schema
schema = {
    'type': 'object',
    'properties': {
        'severity': {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
        'category': {'type': 'string'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1}
    },
    'required': ['severity', 'category', 'confidence'],
    'additionalProperties': False
}
resp = model.chat_json_schema(json_messages, schema, temperature=0.0, max_tokens=200)
print('JSON schema:', resp.choices[0].message.content)

JSON schema: {"severity":"medium","category":"Brute Force Attack","confidence":0.85}


## 5. Batch Chat

In [13]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
    [{'role': 'user', 'content': 'What is a zero-day? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

Batch 0: Phishing is a cybercrime technique where attackers impersonate legitimate entities to deceive individuals into revealing sensitive information, such as passwords or credit card numbers.
Batch 1: Ransomware is a type of malicious software that encrypts a victim's data, demanding payment for the decryption key to restore access.
Batch 2: A zero-day is a software vulnerability that is unknown to the software's developers and for which no official patch or fix is available, making it susceptible to exploitation by attackers.


## 6. Token Usage

In [14]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     30
Completion tokens: 31
Total tokens:      61


## 7. Get Config

In [15]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "gpt-4o",
  "provider": "openai",
  "role": "tamas_baseline",
  "api_based": true,
  "estimated_cost": "$25-50"
}


## Summary

All tests passed if no cells raised exceptions above.